In [12]:
# # imports
# import pandas as pd
# import numpy as np
# import re
# from sklearn.cluster import KMeans

# import torch
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.decomposition import NMF
# from sklearn.cluster import KMeans
# from sklearn.metrics.pairwise import cosine_similarity

# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# from sentence_transformers import SentenceTransformer
# from collections import Counter

import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

### reading in data and cleaning

In [13]:
# data downloaded from online, 2500 bills in bills1.csv
bills1_df = pd.read_csv(
    "../data/bills/bills1.csv",
    skiprows=2
)
bills1_df.head()

,Legislation Number,URL,Congress,Title,Sponsor,Date of Introduction,Committees,Latest Action,Latest Action Date,Number of Cosponsors,Amends Bill,Date Offered,Date Submitted,Date Proposed,Amends Amendment
0,H.R. 7933,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 10, United States Code, to expa...","Watson Coleman, Bonnie [Rep.-D-NJ-12]",03/12/2026,House - Armed Services,Referred to the House Committee on Armed Servi...,03/12/2026,0,NaN,NaN,NaN,NaN,NaN
1,H.R. 7932,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),HONOR Gold Star Families Act,"Van Epps, Matt [Rep.-R-TN-7]",03/12/2026,House - Armed Services,Referred to the House Committee on Armed Servi...,03/12/2026,11,NaN,NaN,NaN,NaN,NaN
2,H.R. 7931,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To amend the Employee Retirement Income Securi...,"Van Drew, Jefferson [Rep.-R-NJ-2]",03/12/2026,House - Education and Workforce,Referred to the House Committee on Education a...,03/12/2026,1,NaN,NaN,NaN,NaN,NaN
3,H.R. 7930,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 18, United States Code, to incr...","Tlaib, Rashida [Rep.-D-MI-12]",03/12/2026,House - Judiciary,Referred to the House Committee on the Judiciary.,03/12/2026,6,NaN,NaN,NaN,NaN,NaN
4,H.R. 7929,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To direct the Secretary of Transportation to i...,"Titus, Dina [Rep.-D-NV-1]",03/12/2026,"House - Science, Space, and Technology","Referred to the House Committee on Science, Sp...",03/12/2026,0,NaN,NaN,NaN,NaN,NaN


In [14]:
# concatenating bills dataframes of 2500 bills each into bills_df
bills_df = bills1_df
# ...

In [ ]:
# investigating csv
print(f'df.shape: {bills_df.shape}')
print(f'df.columns: {bills_df.columns}')
print(f'df.info(): {bills_df.info()}')
print(f'df.describe(include="all"): {bills_df.describe(include="all")}')

df.shape: (2500, 15)
df.columns: Index(['Legislation Number', 'URL', 'Congress', 'Title', 'Sponsor',
       'Date of Introduction', 'Committees', 'Latest Action',
       'Latest Action Date', 'Number of Cosponsors', 'Amends Bill',
       'Date Offered', 'Date Submitted', 'Date Proposed', 'Amends Amendment'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Legislation Number    2500 non-null   object 
 1   URL                   2500 non-null   object 
 2   Congress              2500 non-null   object 
 3   Title                 2500 non-null   object 
 4   Sponsor               2500 non-null   object 
 5   Date of Introduction  2500 non-null   object 
 6   Committees            2500 non-null   object 
 7   Latest Action         2500 non-null   object 
 8   Latest Action Date    2500 non-null   object 


In [16]:
# convert to datetime format
date_cols = [
    "Date of Introduction",
    "Latest Action Date",
    "Date Offered",
    "Date Submitted",
    "Date Proposed"
]
# date range (work in progress, need to get more data)
for col in date_cols:
    bills_df[col] = pd.to_datetime(bills_df[col], errors="coerce")

print(f'min date: {bills_df["Date of Introduction"].min()}')
print(f'max date: {bills_df["Date of Introduction"].max()}')

min date: 2025-09-17 00:00:00
max date: 2026-03-12 00:00:00


In [17]:
# missing vals
missing = bills_df.isnull().sum().sort_values(ascending=False)
print(missing)

Amends Bill             2500
Date Offered            2500
Date Submitted          2500
Date Proposed           2500
Amends Amendment        2500
Legislation Number         0
URL                        0
Congress                   0
Title                      0
Sponsor                    0
Date of Introduction       0
Committees                 0
Latest Action              0
Latest Action Date         0
Number of Cosponsors       0
dtype: int64


In [18]:
# List of columns to drop
drop_cols = [
    "Date Offered",
    "Date Submitted",
    "Date Proposed",
    "Amends Bill",
    "Amends Amendment"
]

bills_df = bills_df.drop(columns=drop_cols)
bills_df.columns

Index(['Legislation Number', 'URL', 'Congress', 'Title', 'Sponsor',
       'Date of Introduction', 'Committees', 'Latest Action',
       'Latest Action Date', 'Number of Cosponsors'],
      dtype='object')

In [19]:
# most frequent committees
print(bills_df["Committees"].value_counts().head())

# most frequent sponsors
print(bills_df["Sponsor"].value_counts().head())

# most frequent latest action
bills_df["Latest Action"].value_counts().head()

Committees
House - Energy and Commerce        277
House - Judiciary                  241
House - Ways and Means             219
House - Education and Workforce    153
House - Financial Services         152
Name: count, dtype: int64
Sponsor
Lawler, Michael [Rep.-R-NY-17]                 33
Vindman, Eugene Simon [Rep.-D-VA-7]            29
Norton, Eleanor Holmes [Del.-D-DC-At Large]    25
Pappas, Chris [Rep.-D-NH-1]                    24
Gottheimer, Josh [Rep.-D-NJ-5]                 23
Name: count, dtype: int64


Latest Action
Referred to the House Committee on Energy and Commerce.        245
Referred to the House Committee on the Judiciary.              231
Referred to the House Committee on Ways and Means.             214
Referred to the House Committee on Education and Workforce.    138
Referred to the House Committee on Financial Services.         130
Name: count, dtype: int64

### cleaning name of sponsor and their info

In [ ]:
# parse sponsor name, party, and state
def parse_sponsor(s):
    match = re.match(r"^(.*?)\s+\[(?:Rep\.|Sen\.|Del\.)-([A-Z])-([A-Z]{2})-", str(s))
    if match:
        name, party_code, state = match.groups()
        party = "Democrat" if party_code == "D" else "Republican" if party_code == "R" else "Independent"
        # split "Last, First" → handle multi-word last names like "Watson Coleman, Bonnie"
        if "," in name:
            last, first = name.split(",", 1)
            last = last.strip()
            first = first.strip()
        else:
            last, first = name.strip(), None
        return first, last, party, state
    return None, s, None, None

bills_df[["Sponsor First Name", "Sponsor Last Name", "Party", "State"]] = bills_df["Sponsor"].apply(
    lambda x: pd.Series(parse_sponsor(x))
)

# drop sponsor col
bills_df = bills_df.drop(columns=["Sponsor"])

# make date of introduction and latest action date into datetime
bills_df["Date of Introduction"] = pd.to_datetime(bills_df["Date of Introduction"])
bills_df["Latest Action Date"] = pd.to_datetime(bills_df["Latest Action Date"])

# strip "House - " or "Senate - " prefix from Committees
bills_df["Committees"] = bills_df["Committees"].str.replace(r"^(House|Senate)\s*-\s*", "", regex=True)

bills_df.head()

,Legislation Number,URL,Congress,Title,Date of Introduction,Committees,Latest Action,Latest Action Date,Number of Cosponsors,Sponsor First Name,Sponsor Last Name,Party,State
0,H.R. 7933,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 10, United States Code, to expa...",2026-03-12,Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,0,Bonnie,Watson Coleman,Democrat,NJ
1,H.R. 7932,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),HONOR Gold Star Families Act,2026-03-12,Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,11,Matt,Van Epps,Republican,TN
2,H.R. 7931,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To amend the Employee Retirement Income Securi...,2026-03-12,Education and Workforce,Referred to the House Committee on Education a...,2026-03-12,1,Jefferson,Van Drew,Republican,NJ
3,H.R. 7930,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 18, United States Code, to incr...",2026-03-12,Judiciary,Referred to the House Committee on the Judiciary.,2026-03-12,6,Rashida,Tlaib,Democrat,MI
4,H.R. 7929,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To direct the Secretary of Transportation to i...,2026-03-12,"Science, Space, and Technology","Referred to the House Committee on Science, Sp...",2026-03-12,0,Dina,Titus,Democrat,NV


### save cleaned data to csv

In [21]:
bills_df.to_csv("../data/bills/bills_cleaned.csv", index=False)

In [23]:
embedder = SentenceTransformer("all-MiniLM-L6-v2",)
title_embeddings = embedder.encode(bills_df["Title"].tolist(), show_progress_bar=True)

NameError: name 'SentenceTransformer' is not defined

In [4]:
#fit KMeans with n = 7
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
bills_df["kmeans_cluster"] = kmeans.fit_predict(title_embeddings)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [5]:
# find and print 5 sentences closest to the centroid of each cluster
print("Discovered topics (KMeans on embeddings):\n")
for c in range(7):
    cluster_mask = bills_df["kmeans_cluster"] == c
    cluster_embs = title_embeddings[cluster_mask]
    centroid = kmeans.cluster_centers_[c]
    dists = cosine_similarity([centroid], cluster_embs)[0]
    top_idx = dists.argsort()[-5:][::-1]
    cluster_reviews = bills_df[cluster_mask].iloc[top_idx]
    print(f"  ── Cluster {c} ({cluster_mask.sum()} titles) ──")
    for _, row in cluster_reviews.iterrows():
        print(f"    \"{row['Title'][:200]}...\"")
    print()

Discovered topics (KMeans on embeddings):

  ── Cluster 0 (330 titles) ──
    "___ Act..."
    "BASIC Act..."
    "REAL Act..."
    "PLAY Act..."
    "MOVE Act..."

  ── Cluster 1 (253 titles) ──
    "Improving Veteran Access to Care Act..."
    "Veteran Benefits Enhancement Act..."
    "CARING for Our Veterans Health Act of 2025..."
    "Connecting Veterans to Care Act of 2025..."
    "Veterans HOPE Act..."

  ── Cluster 2 (269 titles) ──
    "Farm and Family Relief Act..."
    "Rural Housing Regulatory Relief Act..."
    "Empowering Rural Communities Act..."
    "FARMS Act..."
    "Assistance for Rural Water Systems Act of 2026..."

  ── Cluster 3 (220 titles) ──
    "To provide for the transfer to the Office for State and Local Law Enforcement of the Department of Homeland Security of the National Threat Evaluation and Reporting Program of the Department, and for ..."
    "To reauthorize Trade Adjustment Assistance programs, and for other purposes...."
    "To provide the Secretary 

In [6]:
# pick distinct categories

# veterans
# rural
# funding
# infrastructure

# some more i think would be worth being distinct

# education
# healthcare
# environment
# public safety
# housing

In [7]:
topic_descriptions = {
    "veterans":"focused on the rights, benefits, healthcare, housing, or services for military veterans and their families.",
    "rural": "addressing the distinct needs of rural communities, including agriculture, broadband access, rural healthcare, and economic development in non-urban areas.",
    "funding": "primarily allocates, appropriates, or adjusts financial resources for government programs, agencies, or initiatives without being tied to a single policy domain.",
    "infrastructure": "related to the construction, maintenance, or modernization of physical systems such as roads, bridges, utilities, water systems, and public transit.",
    "education":"governing K–12 schools, higher education, student aid, workforce training, or early childhood programs.",
    "healthcare":"addressing public health, insurance coverage, drug pricing, hospital systems, or medical research.",
    "environment":"focused on conservation, climate policy, environmental regulation, or natural resource management.",
    "public_safety":"related to law enforcement, criminal justice reform, emergency management, or disaster response.",
    "housing":"addressing affordable housing, homelessness, zoning, or mortgage and rental assistance."
}

In [8]:
# embed topics
topic_names = list(topic_descriptions.keys())
topic_texts = list(topic_descriptions.values())
topic_embeddings = embedder.encode(topic_texts)


In [10]:
# cosine similarity to find sim between sentence embedding and topic embedding
sim_matrix = cosine_similarity(title_embeddings, topic_embeddings)

# Convert to a DataFrame so we can work with column names instead of indices
sim_df = pd.DataFrame(sim_matrix, columns=topic_names)

In [20]:
# For each row, we collect topic names where the similarity exceeds threshold.
bills_df["topic"] = ""
for i, row in sim_df.iterrows():
    max = 0
    top_col = "Other"
    for col in sim_df.columns:
        if row[col] > max and row[col] > 0.15:
            max = row[col]
            top_col = str(col)

    bills_df.at[i, 'topic'] = top_col
    


In [34]:
bills_df[2496:2497]

,Legislation Number,URL,Congress,Title,Sponsor,Date of Introduction,Committees,Latest Action,Latest Action Date,Number of Cosponsors,kmeans_cluster,topic
2496,H.R. 5437,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),Protection of Lawful Commerce in Stone Slab Pr...,"McClintock, Tom [Rep.-R-CA-5]",2025-09-17,House - Judiciary,Referred to the House Committee on the Judiciary.,2025-09-17,9,5,Other


In [37]:
bills_df = bills_df.drop(columns=["kmeans_cluster"])

In [39]:
bills_df.to_csv("../data/bills/bills_with_topic.csv")